# Dual-Encoder Router — Multi-Dataset Robustness Benchmark (RunPod)

Evaluates the **Dual-Encoder (step) router** — the most promising vocabulary-pruning
approach from the paper — across three new domains:

| Dataset | Domain |
|---|---|
| Penn Treebank (PTB) | Formal news (WSJ) |
| AG News | News headlines |
| CodeParrot | Python source code |

For each dataset a **fresh router is trained from scratch** on that dataset's train
split. Metrics: ΔPPL (pruned − baseline) and token acceptance rate at K ∈ {5k, 10k}.

Results are written to `results/multi_dataset_results.csv` and
`results/multi_dataset_summary.csv`.

## Prerequisites (do these ONCE before launching the pod)

### 1. HuggingFace token
1. Go to https://huggingface.co/settings/tokens → create a token with **Read** scope
2. Accept the Llama-3.2 license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
3. RunPod → **Secrets** → **Add Secret**: Key `HF_TOKEN`, Value: your token (`hf_...`)
4. In your pod template reference it as `{{ RUNPOD_SECRET_HF_TOKEN }}`

### 2. GitHub token
1. GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)
   → scope: **repo**
2. RunPod → **Secrets** → Key `GITHUB_TOKEN`, Value: your PAT (`ghp_...`)
3. In your pod template: `{{ RUNPOD_SECRET_GITHUB_TOKEN }}`

### 3. GPU
- **RTX 4090 (24 GB)** or **A100 40 GB** recommended for 1B
- Llama-3.2-1B-Instruct in bfloat16 requires ~2.5 GB VRAM
- Blackwell GPUs (SM 12.0): Cell 1 auto-detects and reinstalls PyTorch 2.7+

---
**After running:** download `results_multi_dataset.zip` from `/workspace/results_multi_dataset.zip`

In [ ]:
# ── Cell 1: Verify GPU + fix PyTorch/CUDA arch mismatch ────────────────────────
#
# Blackwell GPUs (RTX PRO 4500, RTX 5090, etc. — SM 12.0) require PyTorch ≥ 2.7.
# Older PyTorch wheels lack SM 12.0 cubins and crash with:
#   RuntimeError: CUDA error: no kernel image is available for execution on the device
# This cell detects the mismatch and reinstalls the correct wheel automatically.
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — check pod GPU settings')

import torch
print(f'torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        major, minor = torch.cuda.get_device_capability(i)
        print(f'CUDA device {i}: {name}, {vram:.1f} GB  (SM {major}.{minor})')

# ── Detect SM 12.0+ (Blackwell) with a PyTorch build that predates it ──────────
needs_reinstall = False
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    sm = major * 10 + minor
    torch_major = int(torch.__version__.split('.')[0])
    torch_minor = int(torch.__version__.split('.')[1])
    if sm >= 120 and (torch_major, torch_minor) < (2, 7):
        needs_reinstall = True
        print(f'\n*** SM {major}.{minor} (Blackwell) detected with PyTorch {torch.__version__} ***')
        print('*** PyTorch < 2.7 has no SM 12.0 cubin — reinstalling PyTorch 2.7+ (cu128)... ***\n')

import pathlib
SENTINEL = pathlib.Path('/tmp/.torch_reinstalled')

if SENTINEL.exists():
    SENTINEL.unlink()
    import torch
    print(f'Kernel restarted successfully. PyTorch {torch.__version__} loaded.')
    print('PyTorch/CUDA arch OK — continue with Cell 2.')
elif needs_reinstall:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--force-reinstall', '--upgrade',
        'torch', 'torchvision', 'torchaudio',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
    ])
    r = subprocess.run([sys.executable, '-m', 'pip', 'show', 'torch'],
                       capture_output=True, text=True)
    version_line = next((l for l in r.stdout.splitlines() if l.startswith('Version')), 'Version: unknown')
    print(f'pip show torch → {version_line}')
    SENTINEL.touch()
    print()
    print('=' * 60)
    print('PyTorch 2.7+cu128 force-reinstalled.')
    print()
    print('  *** RESTART THE KERNEL NOW ***')
    print('  Kernel menu → Restart Kernel  (or click the restart button)')
    print('  Then re-run Cell 1 once — it will confirm success.')
    print('=' * 60)
else:
    print('PyTorch/CUDA arch OK — no reinstall needed.')

try:
    import triton
    print(f'Triton: {triton.__version__} ✓  (fast router kernel available)')
except ImportError:
    print('Triton: not installed — PyTorch fast-path will be used instead')

In [ ]:
# ── Cell 2: HuggingFace token + model selection ──────────────────────────────
# RunPod injects secrets as env vars with the RUNPOD_SECRET_ prefix.
import os

hf_token = os.environ.get('RUNPOD_SECRET_HF_TOKEN') or os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError(
        'HF_TOKEN not found.\n'
        'Set it in RunPod Secrets as HF_TOKEN and reference it in the pod template '
        'as {{ RUNPOD_SECRET_HF_TOKEN }}, or set HF_TOKEN directly in the environment.'
    )

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token  # legacy env var
print('HF_TOKEN loaded ✓')

# 1B model — change to Llama-3.1-8B-Instruct for the 8B variant
os.environ['MODEL_NAME'] = 'meta-llama/Llama-3.2-1B-Instruct'
print(f'MODEL_NAME set to: {os.environ["MODEL_NAME"]}')

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# RunPod PyTorch images ship with torch pre-installed; this adds the rest.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print('Dependencies installed ✓')

In [ ]:
# ── Cell 4: Clone repo ───────────────────────────────────────────────────────
# Reads GITHUB_TOKEN from the RunPod secret injected as RUNPOD_SECRET_GITHUB_TOKEN.
import os

gh_token = os.environ.get('RUNPOD_SECRET_GITHUB_TOKEN') or os.environ.get('GITHUB_TOKEN')
if not gh_token:
    raise RuntimeError(
        'GITHUB_TOKEN not found.\n'
        'Set it in RunPod Secrets as GITHUB_TOKEN and reference it in the pod template '
        'as {{ RUNPOD_SECRET_GITHUB_TOKEN }}, or set GITHUB_TOKEN directly in the environment.'
    )

WORKDIR = '/workspace/dsts'
REPO_URL = f'https://{gh_token}@github.com/deil87/dsts.git'

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {WORKDIR} remote set-url origin {REPO_URL}
    !git -C {WORKDIR} pull

%cd {WORKDIR}
print(f'Working directory: {os.getcwd()}')

os.environ['MODEL_NAME'] = 'meta-llama/Llama-3.2-1B-Instruct'
print(f'MODEL_NAME: {os.environ["MODEL_NAME"]}')
!ls -la

In [ ]:
# ── Cell 5: Experiment config ────────────────────────────────────────────────
# Edit these values before running the sweep.
# ─────────────────────────────────────────────────────────────────────────────

# Datasets to evaluate. Options: "ptb", "ag_news", "codeparrot", "wikitext2"
DATASETS = ["ptb", "ag_news", "codeparrot"]

# Shortlist sizes to sweep.
K_VALUES = [5_000, 10_000]

# FAST_MODE: True  → smaller index (2k completions, 1k train pairs) — ~15-20 min/dataset
#            False → full settings (8k completions, 5k train pairs) — ~45-60 min/dataset
FAST_MODE = True

# ─────────────────────────────────────────────────────────────────────────────
import os

os.environ['FAST_MODE'] = '1' if FAST_MODE else '0'
os.environ['K_VALUES']  = ','.join(str(k) for k in K_VALUES)

if FAST_MODE:
    os.environ['N_INDEX_COMPLETIONS'] = '2000'
    os.environ['N_TRAIN_PAIRS']       = '1000'
else:
    os.environ.pop('N_INDEX_COMPLETIONS', None)
    os.environ.pop('N_TRAIN_PAIRS', None)

print(f'Datasets:           {DATASETS}')
print(f'K values:           {K_VALUES}')
print(f'Fast mode:          {FAST_MODE}')
if FAST_MODE:
    print(f'Index completions:  2,000  (fast)')
    print(f'Train pairs:        1,000  (fast)')
    print(f'Est. runtime:       ~{15 * len(DATASETS)}–{20 * len(DATASETS)} min total')
else:
    print(f'Index completions:  8,000  (full)')
    print(f'Train pairs:        5,000  (full)')
    print(f'Est. runtime:       ~{45 * len(DATASETS)}–{60 * len(DATASETS)} min total')

In [ ]:
# ── Cell 6: Run multi-dataset sweep ─────────────────────────────────────────
# Loads the model once, then for each dataset:
#   - builds a completion index from the train split
#   - trains a fresh Dual-Encoder router from scratch
#   - evaluates full-vocab baseline PPL
#   - evaluates pruned PPL and acceptance rate at each K
# Results are checkpointed after each dataset → results/multi_dataset_results.csv
print('=' * 60)
print('DUAL-ENCODER MULTI-DATASET ROBUSTNESS SWEEP')
print('=' * 60)

from run_multi_dataset import run_sweep
df_results = run_sweep(dataset_keys=DATASETS)

In [ ]:
# ── Cell 7: Summary table ────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv('results/multi_dataset_results.csv')

print('\n=== Full results ===')
with pd.option_context(
    'display.max_rows', 50,
    'display.max_columns', 20,
    'display.width', 120,
    'display.float_format', '{:.4f}'.format,
):
    print(df[['dataset_label', 'domain', 'k', 'baseline_ppl',
               'pruned_ppl', 'delta_ppl', 'acceptance_rate']].to_string(index=False))

print('\n=== Best K per dataset (lowest ΔPPL) ===')
summary = (
    df.sort_values('delta_ppl')
      .groupby('dataset_label', sort=False)
      .first()
      .reset_index()
      [['dataset_label', 'domain', 'k', 'baseline_ppl', 'pruned_ppl', 'delta_ppl', 'acceptance_rate']]
)
with pd.option_context('display.float_format', '{:.4f}'.format, 'display.width', 120):
    print(summary.to_string(index=False))

In [ ]:
# ── Cell 8: Plots ────────────────────────────────────────────────────────────
# Generates three figures saved to results/:
#   multi_dataset_delta_ppl.png   — ΔPPL grouped bar chart
#   multi_dataset_acceptance.png  — acceptance rate grouped bar chart
#   multi_dataset_heatmap.png     — ΔPPL heatmap (datasets × K)
print('=' * 60)
print('Generating cross-dataset plots...')
print('=' * 60)

from src.plots import plot_all_multi_dataset
plot_all_multi_dataset('results/multi_dataset_results.csv')

import os
print('\nPlots saved:')
for f in ['multi_dataset_delta_ppl.png', 'multi_dataset_acceptance.png', 'multi_dataset_heatmap.png']:
    path = f'results/{f}'
    if os.path.exists(path):
        print(f'  {path}  ({os.path.getsize(path)/1024:.0f} KB)')

In [ ]:
# ── Cell 9: Package results for download ────────────────────────────────────
# Zips the results/ directory → /workspace/results_multi_dataset.zip
# Download via the RunPod file browser, or:
#   scp root@<pod-ip>:/workspace/results_multi_dataset.zip .
import shutil, os

OUTPUT_ZIP = '/workspace/results_multi_dataset'
shutil.make_archive(OUTPUT_ZIP, 'zip', 'results')

zip_size = os.path.getsize(OUTPUT_ZIP + '.zip') / 1e6
print(f'results_multi_dataset.zip created: {zip_size:.1f} MB  →  {OUTPUT_ZIP}.zip')
print()
print('To download from your local machine:')
print('  scp root@<pod-ip>:/workspace/results_multi_dataset.zip .')
print()
print('Locally, unzip into your results/ folder:')
print('  unzip results_multi_dataset.zip -d results/')
print('  python -c "from src.plots import plot_all_multi_dataset; plot_all_multi_dataset()"')